<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/secuenciador_laser_V7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Esta version incluye el simulador del plan

En esta version falta, añadir funcion para que el Upload respete la secuencia actual posiciones 1 y 2 tal vez 3.



Tenemos que pasar a algorimto genetico y validar metricas de rendimiento contra planeacion manual

In [ ]:
# ==========================================
# INSTALACIÓN DE LIBRERÍAS (Descomentar en Colab)
# ==========================================

!pip install pulp gurobipy openpyxl

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pulp
import sys
import concurrent.futures

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 68.7 MB/s eta 0:00:00


In [ ]:
# PARÁMETROS ESTRATÉGICOS
PESO_URGENCIA = 80
PESO_DENSIDAD = 20
UMBRAL_RECHAZO = 40.0  # Filtro estricto contra el WIP inútil
LISTA_MAQUINAS = ['LASER 01', 'LASER 02', 'LASER 03', 'LASER 04', 'LASER 05', 'LASER 06' , 'LASER 08']

In [ ]:
# ==========================================
# 1. CARGA DE DATOS Y LIMPIEZA INICIAL
# ==========================================
print("Cargando y limpiando datos...")
try:
    df_folios = pd.read_excel('Plan de Corte_07.07_xComponente.xlsx')
    df_ventas = pd.read_excel('Reporte de Cortos_07.07_laser.xlsx')
except FileNotFoundError:
    print("Error: No se encontraron los archivos Excel. Verifica las rutas.")
    sys.exit()



hoy = datetime.now()

# Limpiar espacios en blanco accidentales

df_folios = df_folios.drop(0)

df_folios.columns = ["No","Maquina","Folio","Programa","Material","Cantidad_placas","Surtido","proceso","Bloque",
                     "Numero de parte","Cant. Piezas","Procesos","Corte","Setup","InicioCorte","Fin Corte","Acum",
                     "Nesteo","Fecha","Disponible","Contenedor","terminadas"]

df_ventas = df_ventas.drop(0)

df_ventas.columns = ["OrdenEPS","Año","Sem","Bloque","Critico","Cliente","Ciudad","Producto Terminado",
                     "Fecha","Real","Componente","Ing","Tipo","Transf","Proceso","Pzas","Hrs",
                     "Nesteada","Cls","Placa","Esp","CodigoRadan","Color pintura","Peso","Familia",
                     "Color","Comentario","Nesteos","idNesteos","TiempoCorteHrs","CorteLaser","Plasma","Nivelado","Pulidor","Limpieza","quintar filos",
                     "Inspeccion","Etiqueta especial"]



df_folios['Numero de parte'] = df_folios['Numero de parte'].astype(str).str.strip()
df_ventas['Componente'] = df_ventas['Componente'].astype(str).str.strip()

df_folios["Corte"] = df_folios["Corte"]*60


# Forzamos la columna a ser matemática. Si el EPS arrojó letras o vacíos, los vuelve 'NaN'
df_ventas['Pzas'] = pd.to_numeric(df_ventas['Pzas'], errors='coerce')

# Ahora sí, borramos TODA LA FILA de cualquier orden que no tenga piezas definidas
df_ventas = df_ventas.dropna(subset=['Pzas'])

Cargando y limpiando datos...


In [ ]:
# ==========================================
# 1.2 ADUANA DE INVENTARIO: FILTRO DE MATERIA PRIMA
# ==========================================
print("\nVerificando disponibilidad de Materia Prima en almacén...")

try:
    # 1. Cargar el reporte de inventario
    # Cambia el nombre del archivo al que uses en la realidad
    df_inventario = pd.read_excel('Placa disponible_26.05_10.16.xlsx')

    # Limpiamos espacios en los nombres de las columnas por seguridad
    df_inventario.columns = df_inventario.columns.str.strip()

    # 2. Renombrar las columnas para que coincidan con tu df_folios
    # Asegúrate de cambiar 'Codigo_Material' y 'Laminas_Disponibles'
    # por los nombres reales de las columnas en tu Excel de inventario.
    # El objetivo es que la columna se llame 'Material' para hacer el cruce.
    df_inventario = df_inventario.rename(columns={
        'Placa': 'Material',
        'Inventario': 'Stock_Almacen'
    })

    # 3. Cruzar los folios con el inventario (Left Join)
    # df_folios ya tiene una columna 'Material' que viene de tu sistema de anidado
    total_folios_original = len(df_folios)
    df_folios = pd.merge(df_folios, df_inventario[['Material', 'Stock_Almacen']], on='Material', how='left')

    # Si un material no está en el Excel de inventario, asumimos que hay 0
    df_folios['Stock_Almacen'] = df_folios['Stock_Almacen'].fillna(0)

    # 4. Separar los folios huérfanos (Sin material) para el reporte de rechazos
    folios_sin_material = df_folios[df_folios['Stock_Almacen'] <= 0].copy()

    if not folios_sin_material.empty:
        print(f"⚠️ ALERTA: Se detectaron {len(folios_sin_material)} folios sin materia prima disponible.")
        # Guardamos un reporte para que Compras o el programador CNC sepan por qué se rechazó
        folios_sin_material.to_excel(f"Folios_Rechazados_Falta_Material_{hoy.strftime('%Y%m%d')}.xlsx", index=False)

    # 5. EL FILTRO ESTRICTO: Conservamos SOLO los folios que tienen stock > 0
    df_folios = df_folios[df_folios['Stock_Almacen'] > 0].copy()

    print(f"Folios aprobados por almacén: {len(df_folios)} de {total_folios_original} originales.")

except FileNotFoundError:
    print("⚠️ No se encontró el archivo de Inventario. Se omitirá el filtro de materia prima.")



Verificando disponibilidad de Materia Prima en almacén...
⚠️ No se encontró el archivo de Inventario. Se omitirá el filtro de materia prima.


In [ ]:
# ==========================================
# 1.5 CONSOLIDACIÓN DEL REPORTE DE VENTAS
# ==========================================
df_ventas['Real'] = pd.to_datetime(df_ventas['Real'])
DIAS_VISIBILIDAD = 7
fecha_limite = hoy + timedelta(days=DIAS_VISIBILIDAD) #DUDA

df_ventas_urgentes = df_ventas[df_ventas['Real'] <= fecha_limite].copy()
df_ventas_agrupado = df_ventas_urgentes.groupby('Componente').agg({
    'Pzas': 'sum',
    'Real': 'min'
}).reset_index()

# ==========================================
# 2. CRUCE DE DATOS Y CÁLCULOS A NIVEL PARTE
# ==========================================
df_cruce = pd.merge(df_folios, df_ventas_agrupado, left_on='Numero de parte', right_on='Componente', how='left')

df_cruce['Pzas'] = df_cruce['Pzas'].fillna(0)
df_cruce['Real'] = pd.to_datetime(df_cruce['Real']).fillna(hoy + timedelta(days=100))
df_cruce['Dias_Faltantes'] = (df_cruce['Real'] - hoy).dt.days
df_cruce['Piezas_Utiles'] = df_cruce.apply(lambda x: min(x['Cant. Piezas'], x['Pzas']), axis=1) #DUDA

df_cruce['Ft_Parte'] = np.where(df_cruce['Dias_Faltantes'] <= 0, 2.0, 1.0 / df_cruce['Dias_Faltantes'])
df_cruce['Ft_Parte'] = np.where(df_cruce['Pzas'] == 0, 0, df_cruce['Ft_Parte'])

# ==========================================
# 3. AGRUPACIÓN Y CÁLCULO DE SCORE POR FOLIO
# ==========================================
resumen_folios = df_cruce.groupby(['Folio', 'Maquina', 'Material', 'Corte']).agg(
    Total_Piezas_Folio=('Cant. Piezas', 'sum'),
    Total_Piezas_Utiles=('Piezas_Utiles', 'sum'),
    Ft_Maximo=('Ft_Parte', 'max')
).reset_index()

# Initialize Densidad_Fd with 0 to avoid ZeroDivisionError
resumen_folios['Densidad_Fd'] = 0.0
# Perform division only where 'Total_Piezas_Folio' is not zero and explicitly cast to float64
mask_non_zero = resumen_folios['Total_Piezas_Folio'] != 0
resumen_folios.loc[mask_non_zero, 'Densidad_Fd'] = (
    resumen_folios.loc[mask_non_zero, 'Total_Piezas_Utiles'] /
    resumen_folios.loc[mask_non_zero, 'Total_Piezas_Folio']
).astype(np.float64)

resumen_folios['Score'] = (PESO_URGENCIA * resumen_folios['Ft_Maximo']) + (PESO_DENSIDAD * resumen_folios['Densidad_Fd'])
resumen_folios['Score'] = resumen_folios['Score'].replace(0, 0.001)

# ==========================================
# 4. Filtro
# ==========================================
resumen_folios['Estatus'] = np.where(
    resumen_folios['Score'] >= UMBRAL_RECHAZO,
    'Aprobado - Secuenciar',
    'Rechazado - Re-Nesting'
)
vista_final = resumen_folios.sort_values(by='Score', ascending=False).reset_index(drop=True)
vista_final.to_excel(f"Vista_Final_nesteos_{hoy.strftime('%Y%m%d')}.xlsx", index=False)

# ==========================================
# 4.5 ALERTA DE CORTOS
# ==========================================
partes_programadas = df_folios['Numero de parte'].unique()
piezas_huerfanas = df_ventas_agrupado[~df_ventas_agrupado['Componente'].isin(partes_programadas)].copy()
piezas_huerfanas['Dias_Faltantes'] = (piezas_huerfanas['Real'] - hoy).dt.days
piezas_criticas_sin_folio = piezas_huerfanas[piezas_huerfanas['Dias_Faltantes'] <= 2].sort_values(by='Dias_Faltantes')

if not piezas_criticas_sin_folio.empty:
    print("\n" + "!"*60)
    print("¡ALERTA ROJA! PIEZAS VENCIDAS SIN PROGRAMA DE CORTE (NO ESTÁN EN EL LÁSER)")
    print("!"*60)
    print(piezas_criticas_sin_folio[['Componente', 'Pzas', 'Dias_Faltantes']].to_string(index=False))
    print("*"*60 + "\n")


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
¡ALERTA ROJA! PIEZAS VENCIDAS SIN PROGRAMA DE CORTE (NO ESTÁN EN EL LÁSER)
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
         Componente   Pzas  Dias_Faltantes
           15141946   38.0            -188
           16804314   36.0            -188
         1068313-04    3.0            -154
       RSF-02-00107    6.0            -124
       RSF-02-00108   14.0            -124
       RSF-02-00109   15.0            -124
       RSF-02-00111    4.0            -124
       RSF-02-00101    3.0            -124
       RSF-02-00219    2.0            -124
       RSF-02-00301    1.0            -124
       RSF-02-00113   11.0            -124
       RSF-02-00202    4.0            -124
       RSF-02-00302    5.0            -124
       RSF-02-00402    4.0            -124
       RSF-02-00701    3.0            -124
       RSF-02-00106    6.0            -124
    A340670ITEM03-A    1.0            -124
       RSF-02-00110    7.0  

In [ ]:
# ==========================================
# 5. MOTOR DE OPTIMIZACIÓN (FUNCIÓN AISLADA)
# ==========================================
def optimizar_maquina(nombre_maquina, df_global):
    """
    Función que recibe el nombre de una máquina, filtra sus datos,
    resuelve el modelo matemático y devuelve un DataFrame con la secuencia.
    """
    df_maquina = df_global[(df_global['Estatus'] == 'Aprobado - Secuenciar') &
                           (df_global['Maquina'] == nombre_maquina)].copy()

    # Tomar solo los 15 más críticos para evitar la explosión combinatoria
    df_maquina = df_maquina.sort_values(by='Score', ascending=False).head(20)

    if df_maquina.empty:
        return pd.DataFrame() # Retorna tabla vacía si no hay folios

    # Nodo ficticio 'Actual'
    nodo_actual_df = pd.DataFrame({
        'Folio': ['Actual'], 'Maquina': [nombre_maquina],
        'Material': ['Ninguno'], 'Corte': [0.0], 'Score': [9999.0]
    })

    columnas_necesarias = ['Folio', 'Maquina', 'Material', 'Corte', 'Score']
    df_secuencia = pd.concat([nodo_actual_df, df_maquina[columnas_necesarias]], ignore_index=True)

    nodos = df_secuencia['Folio'].tolist()
    N = len(nodos)

    # Matriz de Setups
    S = np.zeros((N, N))
    materiales = df_secuencia['Material'].tolist()
    for i in range(N):
        for j in range(N):
            if i != j and materiales[i] != materiales[j]:
                S[i][j] = 15.0 # 15 mins por cambio de material
            else:
                S[i][j] = 0.0

    T = {nodos[i]: df_secuencia['Corte'].iloc[i] for i in range(N)}
    P = {nodos[i]: df_secuencia['Score'].iloc[i] for i in range(N)}

    # Modelo MILP
    # Se añade el nombre de la máquina para evitar conflictos en paralelo
    prob = pulp.LpProblem(f"Sec_{nombre_maquina.replace(' ', '_')}", pulp.LpMinimize)

    x = pulp.LpVariable.dicts(f"ruta_{nombre_maquina}", [(nodos[i], nodos[j]) for i in range(N) for j in range(N) if i != j], cat='Binary')
    C = pulp.LpVariable.dicts(f"C_{nombre_maquina}", nodos, lowBound=0, cat='Continuous')

    prob += pulp.lpSum(C[i] / P[i] for i in nodos if i != 'Actual') + 10 * pulp.lpSum(x[(nodos[i], nodos[j])] * S[i][j] for i in range(N) for j in range(N) if i != j)

    for i in nodos:
        prob += pulp.lpSum(x[(i, j)] for j in nodos if i != j) == 1
        prob += pulp.lpSum(x[(j, i)] for j in nodos if i != j) == 1

    # La Gran M dinámica
    M = sum(T.values()) + (15.0 * N)
    for i in range(N):
        for j in range(N):
            if i != j and nodos[j] != 'Actual':
                nodo_i, nodo_j = nodos[i], nodos[j]
                prob += C[nodo_j] >= C[nodo_i] + S[i][j] + T[nodo_j] - M * (1 - x[(nodo_i, nodo_j)])

    prob += C['Actual'] == 0

    # Resolver con Gurobi. msg=False para no saturar la consola con 6 procesos a la vez. TimeLimit=60s. (corre nomral)
    prob.solve(pulp.GUROBI(timeLimit=300, msg=False))

    # Extracción
    if pulp.value(prob.objective) is None:
        print(f"[!] {nombre_maquina}: Gurobi no encontró ruta en el tiempo límite.")
        return pd.DataFrame()

    secuencia_final = []
    nodo_actual = 'Actual'
    for _ in range(N):
        secuencia_final.append(nodo_actual)
        for j in nodos:
            if nodo_actual != j and pulp.value(x[(nodo_actual, j)]) > 0.5:
                nodo_actual = j
                break

    # Construir tabla de resultados para esta máquina
    datos_resultado = []
    tiempo_acumulado = 0
    orden = 1

    for nodo in secuencia_final[1:]: # Omitir dummy
        idx = nodos.index(nodo)
        idx_prev = nodos.index(secuencia_final[secuencia_final.index(nodo)-1])

        setup = S[idx_prev][idx]
        tiempo_acumulado += setup
        inicio = tiempo_acumulado
        tiempo_acumulado += T[nodo]
        material = df_secuencia.loc[df_secuencia['Folio'] == nodo, 'Material'].values[0]
        score = P[nodo]

        datos_resultado.append({
            'Maquina': nombre_maquina,
            'Orden': orden,
            'Folio': nodo,
            'Material': material,
            'Setup (min)': setup,
            'Inicio (min)': inicio,
            'Fin (min)': tiempo_acumulado,
            'Score_Interno': round(score, 2)
        })
        orden += 1

    return pd.DataFrame(datos_resultado)

In [ ]:
# ==========================================
# 6. EJECUCIÓN EN PARALELO (MULTITHREADING)
# ==========================================
print("\n=== INICIANDO OPTIMIZACIÓN EN PARALELO (GUROBI) ===")
resultados_planta = []

# max_workers=6 asegura que intente correr las 6 máquinas al mismo tiempo
with concurrent.futures.ThreadPoolExecutor(max_workers=7) as executor:
    # Lanza las 6 tareas
    futuros = {executor.submit(optimizar_maquina, maq, vista_final): maq for maq in LISTA_MAQUINAS}

    # Recoge los resultados conforme vayan terminando
    for futuro in concurrent.futures.as_completed(futuros):
        maquina = futuros[futuro]
        try:
            df_res = futuro.result()
            if not df_res.empty:
                resultados_planta.append(df_res)
                print(f"[*] ¡Éxito! Secuencia generada para {maquina}")
            else:
                print(f"[-] {maquina} omitida (Sin folios aprobados o sin solución).")
        except Exception as e:
            print(f"[X] Error procesando {maquina}: {e}")


=== INICIANDO OPTIMIZACIÓN EN PARALELO (GUROBI) ===
Restricted license - for non-production use only - expires 2027-11-29
Restricted license - for non-production use only - expires 2027-11-29
[*] ¡Éxito! Secuencia generada para LASER 04
Freeing default Gurobi environment
[*] ¡Éxito! Secuencia generada para LASER 02
[*] ¡Éxito! Secuencia generada para LASER 01
[*] ¡Éxito! Secuencia generada para LASER 03
[*] ¡Éxito! Secuencia generada para LASER 05
[*] ¡Éxito! Secuencia generada para LASER 06
[*] ¡Éxito! Secuencia generada para LASER 08


In [ ]:
# ==========================================
# 7. CONSOLIDACIÓN Y EXPORTACIÓN A EXCEL
# ==========================================
if resultados_planta:
    # Juntar todas las tablas en una sola
    df_planta_completa = pd.concat(resultados_planta, ignore_index=True)

    # Ordenar por Máquina y luego por la Secuencia de Operación
    df_planta_completa = df_planta_completa.sort_values(by=['Maquina', 'Orden']).reset_index(drop=True)

    # Exportar al archivo final
    nombre_archivo_salida = f"Secuencia_Optimizada_Planta_{hoy.strftime('%Y%m%d_%H%M')}.xlsx"
    df_planta_completa.to_excel(nombre_archivo_salida, index=False)

    print("\n" + "="*60)
    print("¡OPTIMIZACIÓN FINALIZADA CON ÉXITO!")
    print(f"El plan de trabajo de toda la planta se ha guardado en: {nombre_archivo_salida}")
    print("="*60)
else:
    print("\nNo se generaron secuencias. Verifica los datos de entrada o el umbral de rechazo.")


¡OPTIMIZACIÓN FINALIZADA CON ÉXITO!
El plan de trabajo de toda la planta se ha guardado en: Secuencia_Optimizada_Planta_20260707_2332.xlsx


In [ ]:
# ==========================================
# 11. VISUALIZACIÓN: CARGA DE TRABAJO (CORTE VS SETUP)
# ==========================================
import plotly.graph_objects as go
import pandas as pd

print("Generando gráfico de carga operativa y setups...")

if 'df_planta_completa' in locals() and not df_planta_completa.empty:

    # 1. Recuperar el tiempo de corte puro
    # El archivo de Gurobi no guarda el corte puro, guarda inicio/fin.
    # Lo calculamos restando: (Fin - Inicio) - Setup = Corte Efectivo
    df_planta_completa['Corte_Efectivo_min'] = (df_planta_completa['Fin (min)'] - df_planta_completa['Inicio (min)'])

    # 2. Agrupar la suma de Setups y suma de Cortes por Máquina
    carga_maquinas = df_planta_completa.groupby('Maquina').agg(
        Total_Setup_min=('Setup (min)', 'sum'),
        Total_Corte_min=('Corte_Efectivo_min', 'sum')
    ).reset_index()

    # 3. Convertir a horas y redondear a 2 decimales
    carga_maquinas['Horas_Setup'] = (carga_maquinas['Total_Setup_min'] / 60).round(2)
    carga_maquinas['Horas_Corte'] = (carga_maquinas['Total_Corte_min'] / 60).round(2)

    # Crear una columna de tiempo total solo para ordenar visualmente el gráfico
    carga_maquinas['Carga_Total'] = carga_maquinas['Horas_Setup'] + carga_maquinas['Horas_Corte']
    carga_maquinas = carga_maquinas.sort_values(by='Carga_Total', ascending=False)

    # 4. Construir el gráfico Apilado (Stacked Bar)
    fig = go.Figure()

    # Trace 1: El tiempo de Valor Agregado (Corte)
    fig.add_trace(go.Bar(
        x=carga_maquinas['Maquina'],
        y=carga_maquinas['Horas_Corte'],
        name='Tiempo de Corte Efectivo',
        marker_color='#00529b', # Azul sólido
        text=carga_maquinas['Horas_Corte'].apply(lambda x: f"{x}h" if x > 0 else ""),
        textposition='inside'
    ))

    # Trace 2: El tiempo de Desperdicio (Setup)
    fig.add_trace(go.Bar(
        x=carga_maquinas['Maquina'],
        y=carga_maquinas['Horas_Setup'],
        name='Tiempo de Preparación (Setup)',
        marker_color='#e74c3c', # Rojo alerta
        text=carga_maquinas['Horas_Setup'].apply(lambda x: f"{x}h" if x > 0 else ""),
        textposition='inside'
    ))

    # 5. Configuración del diseño
    fig.update_layout(
        title=dict(
            text='Carga de Máquinas: Tiempo Efectivo vs Pérdida por Setups',
            font=dict(size=20)
        ),
        xaxis_title='Máquina Láser',
        yaxis_title='Horas',
        barmode='stack', # <- Esto hace que el Rojo se monte sobre el Azul
        template='plotly_white',
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=50, r=50, t=80, b=50)
    )

    # Línea de referencia (Turno Estándar de 8 horas)
    fig.add_hline(
        y=10.0,
        line_dash="dash",
        line_color="black",
        annotation_text="Límite de Turno (8 hrs)",
        annotation_position="bottom right"
    )

    fig.show()
else:
    print("Error: No se encontró la tabla de la planta completa para graficar.")

Generando gráfico de carga operativa y setups...


In [ ]:
# ==========================================
# 8. VISTA EJECUTIVA PARA GERENCIA DE PLANTA
# ==========================================
print("Generando reporte de visibilidad para Gerencia...")

# 1. Cruzamos la secuencia final de las máquinas con nuestro df_cruce original
# df_cruce tiene el detalle de qué números de parte van dentro de cada folio
visibilidad_gerencia = pd.merge(
    df_planta_completa[['Maquina', 'Orden', 'Folio', 'Fin (min)']],
    df_cruce[['Folio', 'Numero de parte', 'Piezas_Utiles', 'Pzas', 'Dias_Faltantes']],
    on='Folio',
    how='inner'
)

# 2. Filtramos la "basura": Solo mostramos las piezas que Ventas realmente pidió (Piezas_Utiles > 0)
visibilidad_gerencia = visibilidad_gerencia[
    (visibilidad_gerencia['Piezas_Utiles'] > 0) &
    (visibilidad_gerencia['Fin (min)'] <= 480) # <-- EL LÍMITE DE TURNO
]

# 3. Agrupamos por Número de Parte por si un mismo componente viene en varios folios distintos
reporte_ejecutivo = visibilidad_gerencia.groupby(['Numero de parte']).agg({
    'Pzas': 'first',                     # Total que pide ventas en el backlog
    'Piezas_Utiles': 'sum',              # Total que el láser va a cortar HOY
    'Dias_Faltantes': 'min',# La urgencia
    'Maquina': lambda x: ', '.join(x.unique()), # En qué máquina(s) se está cortando
    'Fin (min)': 'max'                   # En qué minuto del turno sale la última pieza
}).reset_index()

# 4. Renombrar columnas para que sea amigable para gerencia
reporte_ejecutivo.rename(columns={
    'Pzas': 'Demanda_Total',
    'Piezas_Utiles': 'Cobertura_Turno_Actual',
    'Fin (min)': 'Minuto_Aprox_Entrega'
}, inplace=True)

# 5. Ordenar: Primero lo vencido, luego por volumen
reporte_ejecutivo = reporte_ejecutivo.sort_values(by=['Dias_Faltantes', 'Demanda_Total'], ascending=[True, False])

# Exportar el reporte gerencial
nombre_archivo_gerencia = f"Visibilidad_Gerencia_Láser_{hoy.strftime('%Y%m%d_%H%M')}.xlsx"
reporte_ejecutivo.to_excel(nombre_archivo_gerencia, index=False)

print(f"Reporte ejecutivo guardado en: {nombre_archivo_gerencia}")

Generando reporte de visibilidad para Gerencia...
Reporte ejecutivo guardado en: Visibilidad_Gerencia_Láser_20260707_2332.xlsx


In [ ]:
# ==========================================
# 10. ASIGNACIÓN EN CASCADA Y GENERACIÓN DE INSIGHTS
# ==========================================
import pandas as pd

print("\nAnalizando el impacto comercial del turno...")

# 1. Ordenamos las ventas originales por componente y por fecha de entrega
df_ventas_detalle = df_ventas.sort_values(by=['Componente', 'Real']).copy()

# 2. Convertimos la producción programada de hoy en un diccionario
produccion_hoy = reporte_ejecutivo.set_index('Numero de parte')['Cobertura_Turno_Actual'].to_dict()

# 3. Lógica FIFO: Repartimos la producción línea por línea
lineas_cubiertas = []

for idx, row in df_ventas_detalle.iterrows():
    comp = row['Componente']
    pedidas = row['Pzas']

    if comp in produccion_hoy and produccion_hoy[comp] > 0:
        surtido = min(pedidas, produccion_hoy[comp])
        produccion_hoy[comp] -= surtido

        lineas_cubiertas.append({
            'Cliente': row.get('Cliente', 'Desconocido'),
            'Producto_Terminado': row.get('Producto Terminado', 'S/N'),
            'Componente': comp,
            'Fecha_Entrega': row['Real'].strftime('%Y-%m-%d'),
            'Pedidas': pedidas,
            'Surtidas_Hoy': surtido,
            'Estatus': 'Completada 100%' if surtido == pedidas else f'Parcial ({surtido}/{pedidas})'
        })

df_impacto = pd.DataFrame(lineas_cubiertas)

# ==========================================
# INSIGHTS AUTOMÁTICOS Y EXPORTACIÓN MULTI-PESTAÑA
# ==========================================
if not df_impacto.empty:

    # Insight 1: Avance por Cliente (Resumen)
    df_resumen_cliente = df_impacto.groupby('Cliente')['Surtidas_Hoy'].sum().reset_index()
    df_resumen_cliente.rename(columns={'Surtidas_Hoy': 'Total_Piezas_Turno'}, inplace=True)
    df_resumen_cliente = df_resumen_cliente.sort_values(by='Total_Piezas_Turno', ascending=False)

    cliente_ganador = df_resumen_cliente.iloc[0]['Cliente']
    piezas_ganador = df_resumen_cliente.iloc[0]['Total_Piezas_Turno']

    # Insight 2: Líneas de ERP cerradas
    lineas_completadas = df_impacto[df_impacto['Estatus'] == 'Completada 100%']
    total_lineas_cerradas = len(lineas_completadas)

    print("\n" + "="*50)
    print("INSIGHTS EJECUTIVOS DEL TURNO ")
    print("="*50)

    print(f"\n MAYOR AVANCE POR CLIENTE:")
    print(f"El cliente '{cliente_ganador}' verá el mayor progreso al finalizar el turno, ")
    print(f"recibiendo {int(piezas_ganador)} piezas críticas de sus órdenes.")

    print(f"\n LÍNEAS DE VENTA CUBIERTAS:")
    print(f"La secuencia actual cerrará al 100% un total de {total_lineas_cerradas} líneas de pedido del ERP.")

    print("\n TOP 5 PRODUCTOS TERMINADOS CUBIERTOS HOY:")
    print(df_impacto[['Cliente', 'Producto_Terminado', 'Componente', 'Surtidas_Hoy', 'Estatus']].head(5).to_string(index=False))
    print("="*50 + "\n")

    # ---------------------------------------------------------
    # EXPORTACIÓN A EXCEL CON MÚLTIPLES PESTAÑAS
    # ---------------------------------------------------------
    nombre_archivo_impacto = f"Impacto_Lineas_ERP_{hoy.strftime('%Y%m%d_%H%M')}.xlsx"

    with pd.ExcelWriter(nombre_archivo_impacto, engine='openpyxl') as writer:

        # Pestaña 1: El detalle línea por línea para trazabilidad de piso
        df_impacto.to_excel(writer, sheet_name='Detalle Lineas EPS', index=False)

        # Pestaña 2: El resumen ejecutivo consolidado por cliente
        df_resumen_cliente.to_excel(writer, sheet_name='Resumen por Cliente', index=False)

    print(f"Reporte de impacto exportado con éxito a: {nombre_archivo_impacto}")
    print(f"(Incluye pestaña 1: Detalle operativo | Pestaña 2: Resumen ejecutivo)")


Analizando el impacto comercial del turno...

INSIGHTS EJECUTIVOS DEL TURNO 

 MAYOR AVANCE POR CLIENTE:
El cliente 'Caterpillar Inc.' verá el mayor progreso al finalizar el turno, 
recibiendo 1624 piezas críticas de sus órdenes.

 LÍNEAS DE VENTA CUBIERTAS:
La secuencia actual cerrará al 100% un total de 80 líneas de pedido del ERP.

 TOP 5 PRODUCTOS TERMINADOS CUBIERTOS HOY:
                   Cliente Producto_Terminado   Componente  Surtidas_Hoy             Estatus
   CARRIER MEXICO SA DE CV       09RX400052-F   09RX500062           4.0   Parcial (4.0/6.0)
   CARRIER MEXICO SA DE CV       09RX400053-F   09RX500064           4.0   Parcial (4.0/8.0)
   CARRIER MEXICO SA DE CV       09RX500153-2 09RX500153-2           2.0   Parcial (2.0/4.0)
EZI ORDENES INTERNAS - USD           11140779     16817491          20.0 Parcial (20.0/31.0)
EZI ORDENES INTERNAS - USD           11140779     16875975           5.0  Parcial (5.0/25.0)

Reporte de impacto exportado con éxito a: Impacto_Lineas_ERP

In [ ]:
# ==========================================
# 9. VISUALIZACIÓN: DEMANDA CRÍTICA VS COBERTURA POR CLIENTE
# ==========================================
import plotly.graph_objects as go
import pandas as pd

print("Generando gráfico de cobertura crítica por cliente...")

# 1. Obtener la demanda URGE (Vencida + 7 días) por cliente
# Usamos df_ventas_urgentes que ya se calculó en la Sección 1.5
demanda_cliente = df_ventas_urgentes.groupby('Cliente')['Pzas'].sum().reset_index()
demanda_cliente.rename(columns={'Pzas': 'Demanda_Critica'}, inplace=True)

# 2. Obtener la cobertura del turno actual por cliente desde df_impacto
if 'df_impacto' in locals() and not df_impacto.empty:
    cobertura_cliente = df_impacto.groupby('Cliente')['Surtidas_Hoy'].sum().reset_index()
else:
    cobertura_cliente = pd.DataFrame(columns=['Cliente', 'Surtidas_Hoy'])

# 3. Cruzar ambos datos (Left Join para mantener el backlog urgente)
df_plot_cliente = pd.merge(demanda_cliente, cobertura_cliente, on='Cliente', how='left')

# Convertir cualquier NaN en Surtidas_Hoy a 0
df_plot_cliente['Surtidas_Hoy'] = df_plot_cliente['Surtidas_Hoy'].fillna(0)

# 4. Ordenar para impacto visual:
# Primero los clientes a los que más piezas les vamos a entregar hoy,
# luego por el tamaño de su urgencia total.
df_plot_cliente = df_plot_cliente.sort_values(by=['Surtidas_Hoy', 'Demanda_Critica'], ascending=[False, False]).head(6)

# Convertir a texto por seguridad
df_plot_cliente['Cliente'] = df_plot_cliente['Cliente'].astype(str)

# Iniciar la figura de Plotly
fig = go.Figure()

# Barra de fondo: La Demanda Crítica (Vencido + 7 Días)
fig.add_trace(go.Bar(
    x=df_plot_cliente['Cliente'],
    y=df_plot_cliente['Demanda_Critica'],
    name='Demanda Crítica (Vencido + 7 días)',
    marker_color='#d3d3d3', # Gris claro
    opacity=0.8,
    hoverinfo='x+y+name'
))

# Barra superpuesta: La Cobertura del Turno
fig.add_trace(go.Bar(
    x=df_plot_cliente['Cliente'],
    y=df_plot_cliente['Surtidas_Hoy'],
    name='Producción Turno Actual',
    marker_color='#00529b', # Azul corporativo sólido
    opacity=0.9,
    hoverinfo='x+y+name'
))

# Configuración del diseño
fig.update_layout(
    title=dict(
        text='Plan de Corte vs Demanda Crítica por Cliente',
        font=dict(size=20)
    ),
    xaxis_title='Cliente / Cuenta',
    yaxis_title='Cantidad de Piezas',
    barmode='overlay',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.show()

Generando gráfico de cobertura crítica por cliente...


In [ ]:
# ==========================================
# 9.5 VISUALIZACIÓN: IMPACTO TOTAL DE LA SECUENCIA ÓPTIMA (15 FOLIOS)
# ==========================================
import plotly.graph_objects as go
import pandas as pd

print("\nCalculando impacto comercial para la secuencia completa (sin límite de tiempo)...")

# 1. Reconstruir la producción TOTAL de los 15 folios asignados
if 'df_planta_completa' in locals() and not df_planta_completa.empty:
    df_secuencia_completa = pd.merge(df_planta_completa[['Folio']], df_folios, on='Folio', how='inner')

    # 🟢 EL FIX INTELIGENTE: Agregamos 'Cant. Piezas' a la lista
    posibles_cols_piezas = ['Cant. Piezas', 'Piezas', 'Piezas_Utiles', 'Cantidad', 'Cant', 'Pzas']
    col_piezas = next((col for col in posibles_cols_piezas if col in df_secuencia_completa.columns), None)

    col_parte = 'Numero de parte' if 'Numero de parte' in df_secuencia_completa.columns else 'Componente'

    if not col_piezas:
        print(f"⚠️ Error: No identifiqué la columna de piezas en df_folios.")
        print(f"Tus columnas disponibles son: {df_secuencia_completa.columns.tolist()}")
    else:
        produccion_total = df_secuencia_completa.groupby(col_parte)[col_piezas].sum().to_dict()

        # 2. Lógica FIFO paralela para la Secuencia Completa
        df_ventas_detalle_total = df_ventas.sort_values(by=['Componente', 'Real']).copy()
        lineas_cubiertas_total = []

        for idx, row in df_ventas_detalle_total.iterrows():
            comp = row['Componente']
            pedidas = row['Pzas']

            if comp in produccion_total and produccion_total[comp] > 0:
                surtido = min(pedidas, produccion_total[comp])
                produccion_total[comp] -= surtido

                lineas_cubiertas_total.append({
                    'Cliente': row.get('Cliente', 'Desconocido'),
                    'Surtidas_Total': surtido
                })

        df_impacto_total = pd.DataFrame(lineas_cubiertas_total)

        # 3. Preparar los datos para el gráfico
        demanda_cliente_total = df_ventas_urgentes.groupby('Cliente')['Pzas'].sum().reset_index()
        demanda_cliente_total.rename(columns={'Pzas': 'Demanda_Critica'}, inplace=True)

        if not df_impacto_total.empty:
            cobertura_cliente_total = df_impacto_total.groupby('Cliente')['Surtidas_Total'].sum().reset_index()
        else:
            cobertura_cliente_total = pd.DataFrame(columns=['Cliente', 'Surtidas_Total'])

        df_plot_total = pd.merge(demanda_cliente_total, cobertura_cliente_total, on='Cliente', how='left')
        df_plot_total['Surtidas_Total'] = df_plot_total['Surtidas_Total'].fillna(0)

        df_plot_total = df_plot_total.sort_values(by=['Surtidas_Total', 'Demanda_Critica'], ascending=[False, False]).head(20)
        df_plot_total['Cliente'] = df_plot_total['Cliente'].astype(str)

        # 4. Construir el gráfico
        fig_total = go.Figure()

        fig_total.add_trace(go.Bar(
            x=df_plot_total['Cliente'],
            y=df_plot_total['Demanda_Critica'],
            name='Demanda Crítica (Vencido + 7 días)',
            marker_color='#d3d3d3',
            opacity=0.8,
            hoverinfo='x+y+name'
        ))

        fig_total.add_trace(go.Bar(
            x=df_plot_total['Cliente'],
            y=df_plot_total['Surtidas_Total'],
            name='Producción Secuencia Completa (15 folios)',
            marker_color='#27ae60', # Verde Esmeralda
            opacity=0.9,
            hoverinfo='x+y+name'
        ))

        fig_total.update_layout(
            title=dict(
                text='Proyección de la secuencia: Cobertura Total de la Secuencia Óptima',
                font=dict(size=20)
            ),
            xaxis_title='Cliente / Cuenta',
            yaxis_title='Cantidad de Piezas',
            barmode='overlay',
            template='plotly_white',
            hovermode='x unified',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            ),
            margin=dict(l=50, r=50, t=80, b=50)
        )

        fig_total.show()
else:
    print("Error: No se ha calculado la secuencia de planta completa aún.")


Calculando impacto comercial para la secuencia completa (sin límite de tiempo)...


CARGA Y GENERACION AUTOMATICA

In [ ]:
# ==========================================
# 12. EXPORTACIÓN DE SECUENCIA ÓPTIMA PARA CARGA A ERP
# ==========================================
print("\nPreparando archivo de carga (Upload) para el ERP...")

try:
    # 1. Cargar el reporte original de la secuencia del ERP
    # Cambia este nombre por el archivo real que descargas del sistema
    df_erp_original = pd.read_excel('Plan de Corte053125.xlsx')

    # Guardamos una copia exacta de las columnas originales para el formato final
    columnas_originales = df_erp_original.columns.tolist()

    # Limpiamos espacios en los encabezados por seguridad
    df_erp_original.columns = df_erp_original.columns.str.strip()

    # 2. Extraer la "Guía Maestra" de nuestra optimización
    # Necesitamos el Folio, la nueva Máquina asignada y la hora de inicio para ordenar
    guia_optima = df_planta_completa[['Folio', 'Maquina', 'Inicio (min)']].copy()

    # Renombramos la columna de máquina temporalmente para evitar choques
    guia_optima = guia_optima.rename(columns={'Maquina': 'Maquina_Asignada'})

    # 3. Cruzar ambos archivos usando el 'Folio' como llave (Inner Join)
    # Al usar 'inner', si Gurobi rechazó un folio (por falta de material o bajo score),
    # se elimina automáticamente de este archivo de carga.
    df_upload = pd.merge(df_erp_original, guia_optima, on='Folio', how='inner')

    # 4. Ordenar exactamente como dictó el optimizador
    # Ordenamos primero por Máquina, y luego por el tiempo de inicio
    df_upload = df_upload.sort_values(by=['Maquina_Asignada', 'Inicio (min)'])

    # 5. Sobrescribir la máquina real (por si Gurobi movió el folio a un láser más vacío)
    df_upload['Maquina'] = df_upload['Maquina_Asignada']

    # 6. Recalcular la columna 'No' (La secuencia operativa)
    # Esto reinicia el contador 1, 2, 3... para cada máquina de forma independiente
    df_upload['No'] = df_upload.groupby('Maquina').cumcount() + 1

    # (Opcional) Si el ERP te exige que los campos de "Inicio Corte" y "Fin Corte"
    # también se actualicen, podrías mapearlos aquí desde df_planta_completa.
    # Por ahora los dejamos intactos según tu instrucción para "simplemente cargarlo".

    # 7. Restaurar la estructura y orden original de las columnas
    # Descartamos nuestras columnas auxiliares matemáticas ('Maquina_Asignada', 'Inicio (min)')
    df_upload = df_upload[columnas_originales]

    # 8. Exportar el archivo final listo para el Data Loader del ERP
    nombre_upload = f"Upload_Secuencia_ERP_{hoy.strftime('%Y%m%d_%H%M')}.xlsx"
    df_upload.to_excel(nombre_upload, index=False)

    print(f"✅ Archivo de carga generado exitosamente: {nombre_upload}")
    print(f"El ERP recibirá {len(df_upload)} folios reordenados bajo la nueva secuencia.")

except FileNotFoundError:
    print("⚠️ No se encontró el archivo original del ERP. Omitiendo generación de Upload.")
except KeyError as e:
    print(f"⚠️ Error: El archivo del ERP no contiene la columna esperada: {e}")


Preparando archivo de carga (Upload) para el ERP...
✅ Archivo de carga generado exitosamente: Upload_Secuencia_ERP_20260707_2332.xlsx
El ERP recibirá 132 folios reordenados bajo la nueva secuencia.


In [ ]:
# ==========================================
# 13. SIMULADOR DE ESCENARIOS COMERCIALES (SÚPER REPORTE)
# ==========================================
import pandas as pd
import plotly.graph_objects as go

print("\nCargando módulo de Simulación de Escenarios (Reporte Unificado)...")

def simular_cobertura_turno(archivo_plan_actual, df_ventas, df_ventas_urgentes, horas_turno=8.0):
    """
    Simula la ejecución de un plan de corte usando un reporte de ERP detallado (explosado a nivel pieza).
    """
    minutos_limite = horas_turno * 60.0
    print(f"Iniciando simulación para un turno de {horas_turno} horas ({minutos_limite} min).")

    try:
        # =========================================================
        # 1. Cargar el plan, eliminar encabezado basura y estandarizar
        # =========================================================
        df_plan = pd.read_excel(archivo_plan_actual)

        # Chaleco antibalas: Eliminamos fila combinada del ERP
        df_plan = df_plan.drop(0)

        # Renombramos usando tus 22 columnas exactas
        df_plan.columns = [
            "No", "Maquina", "Folio", "Programa", "Material", "CantPlacas", "Surtido",
            "en proceso", "bloque", "Numero de parte", "Cant_piezas", "ruta de proceso",
            "Corte", "Setup", "iniciocorte", "fincorte", "acum", "nest", "Fecha",
            "disponible", "contenedor", "terminado"
        ]

        # Ajustes de Seguridad y Unidades
        df_plan['No'] = pd.to_numeric(df_plan['No'], errors='coerce')
        # Pasamos el tiempo a minutos y evitamos caídas por celdas vacías
        df_plan['Corte'] = pd.to_numeric(df_plan['Corte'], errors='coerce') * 60.0
        df_plan['Cant_piezas'] = pd.to_numeric(df_plan['Cant_piezas'], errors='coerce').fillna(0)

        # Limpieza de textos y espacios fantasma
        df_plan['Folio'] = df_plan['Folio'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        df_plan['Maquina'] = df_plan['Maquina'].astype(str).str.strip()
        df_plan['Material'] = df_plan['Material'].astype(str).str.strip()
        df_plan['Numero de parte'] = df_plan['Numero de parte'].astype(str).str.strip()

        df_plan = df_plan.dropna(subset=['No', 'Folio'])
        df_plan = df_plan.sort_values(by=['Maquina', 'No'])

        # =========================================================
        # 2. Simular el "Reloj" de la Planta (Lógica de Tiempos)
        # =========================================================
        # Como ya no hay merge, iteramos directamente sobre el archivo limpio
        resultados_sim = []

        for maquina, grupo_maquina in df_plan.groupby('Maquina', sort=False):
            reloj = 0.0
            material_previo = None

            # Sub-agrupamos por Folio para avanzar el reloj solo 1 vez por lámina
            for folio, grupo_folio in grupo_maquina.groupby('Folio', sort=False):

                row_folio = grupo_folio.iloc[0]
                material_actual = row_folio['Material']
                tiempo_corte = row_folio['Corte']

                setup = 15.0 if material_previo is not None and material_actual != material_previo else 0.0

                inicio = reloj + setup
                fin = inicio + tiempo_corte

                # Registramos todas las piezas que vienen en esta lámina
                for idx, row_pieza in grupo_folio.iterrows():
                    resultados_sim.append({
                        'Folio': folio,
                        'Maquina': maquina,
                        'Inicio (min)': inicio,
                        'Fin (min)': fin,
                        'Numero de parte': row_pieza['Numero de parte'],
                        'Piezas_Simuladas': row_pieza['Cant_piezas']
                    })

                reloj = fin
                material_previo = material_actual

        df_reloj_simulado = pd.DataFrame(resultados_sim)

        # =========================================================
        # 3. Lógica FIFO Comercial (Filtro de Turno)
        # =========================================================
        df_dentro_turno = df_reloj_simulado[df_reloj_simulado['Fin (min)'] <= minutos_limite]
        produccion_simulada = df_dentro_turno.groupby('Numero de parte')['Piezas_Simuladas'].sum().to_dict()

        df_ventas_detalle = df_ventas.sort_values(by=['Componente', 'Real']).copy()
        lineas_simuladas = []

        for idx, row in df_ventas_detalle.iterrows():
            comp = row['Componente']
            pedidas = row['Pzas']

            if comp in produccion_simulada and produccion_simulada[comp] > 0:
                surtido = min(pedidas, produccion_simulada[comp])
                produccion_simulada[comp] -= surtido

                lineas_simuladas.append({
                    'Cliente': row.get('Cliente', 'Desconocido'),
                    'Surtidas_Sim': surtido
                })

        df_impacto_sim = pd.DataFrame(lineas_simuladas)

        # =========================================================
        # 4. Preparar Datos y Construir Gráfico
        # =========================================================
        demanda_cliente = df_ventas_urgentes.groupby('Cliente')['Pzas'].sum().reset_index()
        demanda_cliente.rename(columns={'Pzas': 'Demanda_Critica'}, inplace=True)

        if not df_impacto_sim.empty:
            cobertura_cliente = df_impacto_sim.groupby('Cliente')['Surtidas_Sim'].sum().reset_index()
        else:
            cobertura_cliente = pd.DataFrame(columns=['Cliente', 'Surtidas_Sim'])

        df_plot = pd.merge(demanda_cliente, cobertura_cliente, on='Cliente', how='left')
        df_plot['Surtidas_Sim'] = df_plot['Surtidas_Sim'].fillna(0)
        df_plot = df_plot.sort_values(by=['Surtidas_Sim', 'Demanda_Critica'], ascending=[False, False]).head(20)
        df_plot['Cliente'] = df_plot['Cliente'].astype(str)

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=df_plot['Cliente'],
            y=df_plot['Demanda_Critica'],
            name='Demanda Crítica (Vencido + 7 días)',
            marker_color='#d3d3d3',
            opacity=0.8,
            hoverinfo='x+y+name'
        ))

        fig.add_trace(go.Bar(
            x=df_plot['Cliente'],
            y=df_plot['Surtidas_Sim'],
            name=f'Simulación (Turno de {horas_turno} hrs)',
            marker_color='#e67e22',
            opacity=0.9,
            hoverinfo='x+y+name'
        ))

        fig.update_layout(
            title=dict(
                text=f'Simulador: Plan de Corte ({horas_turno} Horas)',
                font=dict(size=20)
            ),
            xaxis_title='Cliente / Cuenta',
            yaxis_title='Cantidad de Piezas',
            barmode='overlay',
            template='plotly_white',
            hovermode='x unified',
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=80, b=50)
        )

        fig.show()

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo de simulación '{archivo_plan_actual}'.")
    except Exception as e:
        print(f"Error inesperado en la simulación: {e}")

# ==========================================
# EJECUCIÓN DEL SIMULADOR
# ==========================================
# Fíjate que eliminamos "df_folios_maestro" de los parámetros porque ya no se necesita.

simular_cobertura_turno(
    archivo_plan_actual='Plan de Corte por numero de parte_26.05_13.13.xlsx',
    df_ventas=df_ventas,
    df_ventas_urgentes=df_ventas_urgentes,
    horas_turno=8.0
)


Cargando módulo de Simulación de Escenarios (Reporte Unificado)...
Iniciando simulación para un turno de 8.0 horas (480.0 min).


In [ ]:
# ==========================================
# 13. SIMULADOR DE ESCENARIOS CON GENERACIÓN DE INSIGHTS
# ==========================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go

print("\nCargando módulo de Simulación de Escenarios con Insights Avanzados...")

def simular_cobertura_turno(archivo_plan_actual, df_ventas, df_ventas_urgentes, horas_turno=8.0):
    """
    Simula el plan, ejecuta la cascada FIFO y genera un reporte analítico de insights comerciales.
    """
    minutos_limite = horas_turno * 60.0

    try:
        # =========================================================
        # 1. Cargar el plan, eliminar encabezado basura y estandarizar
        # =========================================================
        df_plan = pd.read_excel(archivo_plan_actual)
        df_plan = df_plan.drop(0)

        df_plan.columns = [
            "No", "Maquina", "Folio", "Programa", "Material", "CantPlacas", "Surtido",
            "en proceso", "bloque", "Numero de parte", "Cant_piezas", "ruta de proceso",
            "Corte", "Setup", "iniciocorte", "fincorte", "acum", "nest", "Fecha",
            "disponible", "contenedor", "terminado"
        ]

        df_plan['No'] = pd.to_numeric(df_plan['No'], errors='coerce')
        df_plan['Corte'] = pd.to_numeric(df_plan['Corte'], errors='coerce') * 60.0
        df_plan['Cant_piezas'] = pd.to_numeric(df_plan['Cant_piezas'], errors='coerce').fillna(0)

        df_plan['Folio'] = df_plan['Folio'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        df_plan['Maquina'] = df_plan['Maquina'].astype(str).str.strip()
        df_plan['Material'] = df_plan['Material'].astype(str).str.strip()
        df_plan['Numero de parte'] = df_plan['Numero de parte'].astype(str).str.strip()

        df_plan = df_plan.dropna(subset=['No', 'Folio'])
        df_plan = df_plan.sort_values(by=['Maquina', 'No'])

        # =========================================================
        # 2. Simular el "Reloj" de la Planta (Lógica de Tiempos)
        # =========================================================
        resultados_sim = []
        for maquina, grupo_maquina in df_plan.groupby('Maquina', sort=False):
            reloj = 0.0
            material_previo = None

            for folio, grupo_folio in grupo_maquina.groupby('Folio', sort=False):
                row_folio = grupo_folio.iloc[0]
                material_actual = row_folio['Material']
                tiempo_corte = row_folio['Corte']

                setup = 15.0 if material_previo is not None and material_actual != material_previo else 0.0

                inicio = reloj + setup
                fin = inicio + tiempo_corte

                for idx, row_pieza in grupo_folio.iterrows():
                    resultados_sim.append({
                        'Folio': folio,
                        'Maquina': maquina,
                        'Inicio (min)': inicio,
                        'Fin (min)': fin,
                        'Numero de parte': row_pieza['Numero de parte'],
                        'Piezas_Simuladas': row_pieza['Cant_piezas']
                    })
                reloj = fin
                material_previo = material_actual

        df_reloj_simulado = pd.DataFrame(resultados_sim)

        # =========================================================
        # 3. Lógica FIFO Comercial (Filtro de Turno y Cascada)
        # =========================================================
        df_dentro_turno = df_reloj_simulado[df_reloj_simulado['Fin (min)'] <= minutos_limite]
        produccion_simulada = df_dentro_turno.groupby('Numero de parte')['Piezas_Simuladas'].sum().to_dict()

        # Guardamos una copia para calcular el Backlog final
        produccion_disponible_backlog = produccion_simulada.copy()

        df_ventas_detalle = df_ventas.sort_values(by=['Componente', 'Real']).copy()
        lineas_simuladas = []

        for idx, row in df_ventas_detalle.iterrows():
            comp = row['Componente']
            pedidas = row['Pzas']

            if comp in produccion_simulada and produccion_simulada[comp] > 0:
                surtido = min(pedidas, produccion_simulada[comp])
                produccion_simulada[comp] -= surtido

                lineas_simuladas.append({
                    'Cliente': row.get('Cliente', 'Desconocido'),
                    'Producto_Terminado': row.get('Producto Terminado', 'N/A'),
                    'Componente': comp,
                    'Fecha_Entrega': row['Real'],
                    'Pedidas': pedidas,  # 🟢 FIX: Nombre exacto que necesitas
                    'Surtidas_Sim': surtido
                })

        df_impacto_sim = pd.DataFrame(lineas_simuladas)

        # =========================================================
        # 🔴 REPLICACIÓN SECCIÓN 10: GENERACIÓN DE INSIGHTS (KPIs)
        # =========================================================
        print("\n" + "="*50)
        print(f"   REPORTES DE INSIGHTS: SIMULACIÓN ({horas_turno} HORAS)")
        print("="*50)

        # KPI 1: Totales de producción virtual
        total_pzas_solicitadas = df_ventas_urgentes['Pzas'].sum()
        total_pzas_cortadas_turno = df_dentro_turno['Piezas_Simuladas'].sum()

        # KPI 2: Totales asignados eficientemente a urgencias
        if not df_impacto_sim.empty:
            total_pzas_utiles_comercial = df_impacto_sim['Surtidas_Sim'].sum()
        else:
            total_pzas_utiles_comercial = 0

        wip_inutil_sobreproduccion = max(0, total_pzas_cortadas_turno - total_pzas_utiles_comercial)
        nivel_servicio_global = (total_pzas_utiles_comercial / total_pzas_solicitadas * 100) if total_pzas_solicitadas > 0 else 0

        print(f"📊 Demanda Crítica Total de Ventas: {total_pzas_solicitadas:,} piezas")
        print(f"🏭 Piezas Físicas Cortadas en Láser: {total_pzas_cortadas_turno:,} piezas")
        print(f"✅ Piezas Útiles (Asignadas a Clientes Críticos): {total_pzas_utiles_comercial:,} piezas")
        print(f"⚠️ WIP / Sobreproducción (Piezas cortadas que nadie necesita hoy): {wip_inutil_sobreproduccion:,} piezas")
        print(f"📈 Nivel de Servicio Crítico Global de este Plan: {nivel_servicio_global:.2f}%")
        print("-"*50)

        # KPI 3: Desglose Analítico por Cliente
        demanda_cliente = df_ventas_urgentes.groupby('Cliente')['Pzas'].sum().reset_index()
        demanda_cliente.rename(columns={'Pzas': 'Demanda_Critica'}, inplace=True)

        if not df_impacto_sim.empty:
            cobertura_cliente = df_impacto_sim.groupby('Cliente')['Surtidas_Sim'].sum().reset_index()
        else:
            cobertura_cliente = pd.DataFrame(columns=['Cliente', 'Surtidas_Sim'])

        df_plot = pd.merge(demanda_cliente, cobertura_cliente, on='Cliente', how='left')
        df_plot['Surtidas_Sim'] = df_plot['Surtidas_Sim'].fillna(0)
        df_plot['%_Fulfillment'] = (df_plot['Surtidas_Sim'] / df_plot['Demanda_Critica'] * 100).fillna(0)

        print("📋 TOP 5 CLIENTES AFECTADOS / BENEFICIADOS EN ESTE ESCENARIO:")
        df_top_clientes = df_plot.sort_values(by='%_Fulfillment', ascending=False).head(5)
        for _, r in df_top_clientes.iterrows():
            print(f"  • {r['Cliente']}: {r['Surtidas_Sim']:,} / {r['Demanda_Critica']:,} piezas surtidas ({r['%_Fulfillment']:.1f}%)")
        print("="*50)


        print("📋 TOP 5 CLIENTES AFECTADOS / BENEFICIADOS EN ESTE ESCENARIO:")
        df_top_clientes = df_plot.sort_values(by='%_Fulfillment', ascending=False).head(5)
        for _, r in df_top_clientes.iterrows():
            print(f"  • {r['Cliente']}: {r['Surtidas_Sim']:,} / {r['Demanda_Critica']:,} piezas surtidas ({r['%_Fulfillment']:.1f}%)")
        print("="*50)

       # =========================================================
        # 🔴 EXPORTACIÓN DEL REPORTE DE LÍNEAS AL ERP/VENTAS
        # =========================================================
        from datetime import datetime
        hoy_str = datetime.now().strftime('%Y%m%d_%H%M')
        nombre_excel = f"Impacto_Lineas_Simulador_{horas_turno}hrs_{hoy_str}.xlsx"

        if not df_impacto_sim.empty:
            # 🟢 FIX: Agregamos 'Pedidas' a la lista para forzar su aparición en el Excel
            columnas_exportacion = ['Cliente', 'Producto_Terminado', 'Componente', 'Fecha_Entrega', 'Pedidas', 'Surtidas_Sim']

            df_export = df_impacto_sim[columnas_exportacion].copy()
            df_export['Fecha_Entrega'] = df_export['Fecha_Entrega'].dt.strftime('%Y-%m-%d')

            df_export.to_excel(nombre_excel, index=False)
            print(f"💾 Reporte de líneas detallado exportado exitosamente: {nombre_excel}")
        else:
            print("⚠️ No se generó archivo Excel porque ninguna pieza simulada cubrió la demanda.")

        print("="*50)

        # =========================================================
        # 4. Construir Gráfico
        # =========================================================

        # =========================================================
        # 4. Construir Gráfico
        # =========================================================
        df_plot_grafico = df_plot.sort_values(by=['Surtidas_Sim', 'Demanda_Critica'], ascending=[False, False]).head(20)
        df_plot_grafico['Cliente'] = df_plot_grafico['Cliente'].astype(str)

        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=df_plot_grafico['Cliente'], y=df_plot_grafico['Demanda_Critica'],
            name='Demanda Crítica', marker_color='#d3d3d3', opacity=0.8, hoverinfo='x+y+name'
        ))
        fig.add_trace(go.Bar(
            x=df_plot_grafico['Cliente'], y=df_plot_grafico['Surtidas_Sim'],
            name=f'Simulación ({horas_turno} hrs)', marker_color='#e67e22', opacity=0.9, hoverinfo='x+y+name'
        ))

        fig.update_layout(
            title=dict(text=f'Simulador del Plan de Corte ({horas_turno} Horas)', font=dict(size=20)),
            xaxis_title='Cliente / Cuenta', yaxis_title='Cantidad de Piezas', barmode='overlay', template='plotly_white',
            hovermode='x unified', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig.show()

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo de simulación '{archivo_plan_actual}'.")
    except Exception as e:
        print(f"Error inesperado en la simulación: {e}")


Cargando módulo de Simulación de Escenarios con Insights Avanzados...


In [ ]:
# ==========================================
# EJECUCIÓN DEL SIMULADOR
# ==========================================
# Fíjate que eliminamos "df_folios_maestro" de los parámetros porque ya no se necesita.

simular_cobertura_turno(
    archivo_plan_actual='porcomponente_26.05_13.45_laser.xlsx',
    df_ventas=df_ventas,
    df_ventas_urgentes=df_ventas_urgentes,
    horas_turno=8.0
)


   REPORTES DE INSIGHTS: SIMULACIÓN (8.0 HORAS)
📊 Demanda Crítica Total de Ventas: 45,360.0 piezas
🏭 Piezas Físicas Cortadas en Láser: 3,321 piezas
✅ Piezas Útiles (Asignadas a Clientes Críticos): 3,084.0 piezas
⚠️ WIP / Sobreproducción (Piezas cortadas que nadie necesita hoy): 237.0 piezas
📈 Nivel de Servicio Crítico Global de este Plan: 6.80%
--------------------------------------------------
📋 TOP 5 CLIENTES AFECTADOS / BENEFICIADOS EN ESTE ESCENARIO:
  • Dematic Corp.: 91.0 / 302.0 piezas surtidas (30.1%)
  • CNH Industrial America LLC.: 1,086.0 / 5,039.0 piezas surtidas (21.6%)
  • CATERPILLAR INDUSTRIAS MEXICO S DE RL DE CV: 343.0 / 1,721.0 piezas surtidas (19.9%)
  • Caterpillar Inc.: 934.0 / 4,976.0 piezas surtidas (18.8%)
  • CARRIER MEXICO SA DE CV: 60.0 / 542.0 piezas surtidas (11.1%)
📋 TOP 5 CLIENTES AFECTADOS / BENEFICIADOS EN ESTE ESCENARIO:
  • Dematic Corp.: 91.0 / 302.0 piezas surtidas (30.1%)
  • CNH Industrial America LLC.: 1,086.0 / 5,039.0 piezas surtidas (21.6%)
